In [1]:
# Let's calculate the correlation of continuation metrics between tacotron2 and vits.
# We have PPL, VERT, and LLM-as-a-Judge score.

In [4]:
# First, load settings with the same N-K-temperature

tacotron_settings = set()
vits_settings = set()
with open("csv/continuation_result_10s.csv") as f:
    for line in f:
        setting, _, _, _, temperature, _ = line.split(",", 5)
        if "tacotron2" not in setting and "vits" not in setting:
            continue
        if not temperature:
            continue
        NK = setting.split("-", 1)[1]
        if "tacotron2" in setting:
            tacotron_settings.add(f"{NK}-{temperature}")
        elif "vits" in setting:
            vits_settings.add(f"{NK}-{temperature}")
common_settings = tacotron_settings & vits_settings
print("tacotron2 settings:", len(tacotron_settings))
print("vits settings:", len(vits_settings))
print("common settings:", len(common_settings))
# These settings shares same unit sequence
common_settings = sorted(list(common_settings))
print(common_settings)

tacotron2 settings: 21
vits settings: 20
common settings: 11
['20-128-0.6', '20-2048-0.6', '20-256-0.6', '20-4096-0.6', '20-512-0.6', '40-1024-0.7', '40-256-0.6', '40-4096-0.7', '80-4096-0.5', '80-512-0.7', '80-8192-0.7']


In [17]:
# Calculate correlation of ppl scores for common settings
from scipy.stats import spearmanr

tacotron2_ppls = []
vits_ppls = []
for setting in common_settings:
    with open(f"csv/ppl_10s/tacotron2/{setting}.txt") as f, open(f"csv/ppl_10s/vits/{setting}.txt") as g:
        for line_t, line_v in zip(f, g):
            wav_id_t, ppl_t = line_t.strip().split("|", 1)
            wav_id_v, ppl_v = line_v.strip().split("|", 1)
            if wav_id_t == "PPL_corpus":
                continue
            assert wav_id_t == wav_id_v
            tacotron2_ppls.append(float(ppl_t))
            vits_ppls.append(float(ppl_v))

print(len(tacotron2_ppls), len(vits_ppls))
correlation, p_value = spearmanr(tacotron2_ppls, vits_ppls)
print(f"Spearman correlation coefficient: {correlation:.4f} (p-value: {p_value:.4e})")

2274 2274
Spearman correlation coefficient: 0.7569 (p-value: 0.0000e+00)


In [18]:
# Calculate correlation of vert scores for common settings
from asr_eval_bleu import all_scores

tacotron2_verts = []
vits_verts = []
for setting in common_settings:
    all_scores_t = all_scores(f"transcription_cut_10s/tacotron2/{setting}.txt")
    all_scores_v = all_scores(f"transcription_cut_10s/vits/{setting}.txt")
    tacotron2_verts.extend(all_scores_t["VERT"])
    vits_verts.extend(all_scores_v["VERT"])

print(len(tacotron2_verts), len(vits_verts))
correlation, p_value = spearmanr(tacotron2_verts, vits_verts)
print(f"Spearman correlation coefficient: {correlation:.4f} (p-value: {p_value:.4e})")

/work/gk77/k77035/espnet/tools/venv/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/work/gk77/k77035/espnet/tools/venv/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/work/gk77/k77035/espnet/tools/venv/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consid

2274 2274
Spearman correlation coefficient: 0.9024 (p-value: 0.0000e+00)


In [ ]:
# Calculate correlation of llm-as-a-judge scores for common settings

from collections import defaultdict
import itertools
from pathlib import Path
from scipy.stats import spearmanr

from myutils import extract_llm_scores


def id_to_score_dict(lines):
    score_dict = {}
    for line in lines[1:]: # skip header
        wav_id, score, _ = line.strip().split(",", 2)
        score_dict[wav_id] = float(score)
    return score_dict

tacotron2_llmscores = []
vits_llmscores = []
score_dict = defaultdict(lambda: defaultdict(list))
for setting_X, setting_Y in itertools.product(common_settings, repeat=2):
    # {model}-{N}-{K}-{temperature}_vs_{model}-{N}-{K}-{temperature}
    tacotron2_path = Path("pairwise_10s") / f"tacotron2-{setting_X}_vs_tacotron2-{setting_Y}"
    vits_path = Path("pairwise_10s") / f"vits-{setting_X}_vs_vits-{setting_Y}"
    with open(tacotron2_path / "summary.txt") as f, open(vits_path / "summary.txt") as g:
        tacotron2_lines = f.readlines()
        vits_lines = g.readlines() 
    tacotron2_score_dict = id_to_score_dict(tacotron2_lines)
    vits_score_dict = id_to_score_dict(vits_lines)

    # Ensure both dictionaries have the same keys (common wav_ids)
    common_wav_ids = set(tacotron2_score_dict.keys()) & set(vits_score_dict.keys())
    # print(len(common_wav_ids), "common wav ids for", setting_X, setting_Y)
    tacotron2_llmscores.extend([tacotron2_score_dict[wav_id] for wav_id in common_wav_ids])
    vits_llmscores.extend([vits_score_dict[wav_id] for wav_id in common_wav_ids])

print(len(tacotron2_llmscores), len(vits_llmscores))
correlation, p_value = spearmanr(tacotron2_llmscores, vits_llmscores)
print(f"Spearman correlation coefficient: {correlation:.4f} (p-value: {p_value:.4e})")

206 common wav ids for 20-128-0.6 20-128-0.6
205 common wav ids for 20-128-0.6 20-2048-0.6
205 common wav ids for 20-128-0.6 20-256-0.6
206 common wav ids for 20-128-0.6 20-4096-0.6
206 common wav ids for 20-128-0.6 20-512-0.6
206 common wav ids for 20-128-0.6 40-1024-0.7
206 common wav ids for 20-128-0.6 40-256-0.6
206 common wav ids for 20-128-0.6 40-4096-0.7
206 common wav ids for 20-128-0.6 80-4096-0.5
206 common wav ids for 20-128-0.6 80-512-0.7
206 common wav ids for 20-128-0.6 80-8192-0.7
205 common wav ids for 20-2048-0.6 20-128-0.6
206 common wav ids for 20-2048-0.6 20-2048-0.6
205 common wav ids for 20-2048-0.6 20-256-0.6
206 common wav ids for 20-2048-0.6 20-4096-0.6
206 common wav ids for 20-2048-0.6 20-512-0.6
206 common wav ids for 20-2048-0.6 40-1024-0.7
206 common wav ids for 20-2048-0.6 40-256-0.6
206 common wav ids for 20-2048-0.6 40-4096-0.7
206 common wav ids for 20-2048-0.6 80-4096-0.5
206 common wav ids for 20-2048-0.6 80-512-0.7
206 common wav ids for 20-2048-0.6